# Starter analysis notebook

> **Template only. No results are included.** Generated project data is synthetic and must not be presented as real customer evidence.

Before using experiment data, freeze the plan in `PRE_REGISTRATION.md`, confirm provenance and grain, run the SRM check, and validate event logging.

## 1. Load and label data

The example input is synthetic. Replace it only with an approved, validated customer-level dataset and document its provenance.

In [5]:
import pandas as pd
from pathlib import Path
import sys

project_root = Path.cwd()
if project_root.name == 'analysis':
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from analysis.stats import two_proportion_z_test, difference_ci, proportion_ci

path = Path('data/synthetic_experiment.csv')  # synthetic illustration only
path = path if path.is_absolute() else project_root / path
df = pd.read_csv(path, comment='#')
assert df['data_status'].eq('SYNTHETIC').all(), 'Confirm dataset provenance before analysis'
df.head()

,experiment_id,variant_id,customer_id,variant,eligible,assigned_at,exposed,exposure_count,converted,order_count,revenue,discount_cost,contribution_margin,refund,payment_failed,support_contact,device,country,category,data_status
0,301,3012,synthetic_customer_0000001,treatment,True,2026-10-26T00:00:00Z,True,1,False,0,0.0,0.0,0.0,False,False,True,desktop,CA,home,SYNTHETIC
1,301,3012,synthetic_customer_0000002,treatment,True,2026-10-26T00:00:00Z,True,1,False,0,0.0,0.0,0.0,False,False,True,mobile,IN,apparel,SYNTHETIC
2,301,3011,synthetic_customer_0000003,control,True,2026-10-26T00:00:00Z,True,1,False,0,0.0,0.0,0.0,False,False,False,desktop,GB,apparel,SYNTHETIC
3,301,3011,synthetic_customer_0000004,control,True,2026-10-26T00:00:00Z,True,1,True,1,2500.0,0.0,1000.0,False,False,False,desktop,US,home,SYNTHETIC
4,301,3011,synthetic_customer_0000005,control,True,2026-10-26T00:00:00Z,True,1,True,1,2500.0,150.0,850.0,False,False,False,desktop,CA,apparel,SYNTHETIC


## 2. Validate grain and assignment

Confirm one row per customer, only expected variants, and eligibility rules. On real data, run the SRM and exposure SQL before computing outcomes. Stop if duplicate customer rows or assignment anomalies remain.

In [6]:
assert df['customer_id'].is_unique, 'Expected one row per randomized customer'
assert set(df['experiment_id'].unique()) == {301}, 'Unexpected experiment ID'
assert set(df['variant'].unique()) <= {'control', 'treatment'}
expected_ids = {'control': 3011, 'treatment': 3012}
assert df.groupby('variant')['variant_id'].nunique().eq(1).all()
assert df.groupby('variant')['variant_id'].first().to_dict() == expected_ids
df.groupby('variant').size()

variant
control      7561
treatment    7439
dtype: int64

## 3. Primary ITT conversion analysis

The following summaries are descriptive. Interpret a test only after verifying the preregistered sample size, window, MDE, and guardrails.

In [7]:
summary = df.groupby('variant')['converted'].agg(['sum', 'count'])
summary

,sum,count
variant,,
control,1432,7561
treatment,1522,7439


In [8]:
# Control: 1,432 purchases out of 7,561 customers
x0, n0 = 1432, 7561

# Treatment: 1,522 purchases out of 7,439 customers
x1, n1 = 1522, 7439

result = two_proportion_z_test(x0, n0, x1, n1)
ci = difference_ci(x0, n0, x1, n1)

print(result, ci)

ZTestResult(control_rate=0.18939293744213728, treatment_rate=0.20459739212259712, difference=0.015204454680459834, z_statistic=2.3411902211805864, p_value_two_sided=0.01922237053544749) (0.002475117641849312, 0.027933791719070355)


## 4. Economics, guardrails, segments, and decision

### Descriptive summary: contribution margin per assigned customer

In [11]:
economics = (
    df.groupby('variant')['contribution_margin']
      .agg(mean_per_customer='mean', total='sum', customers='count')
)
economics

,mean_per_customer,total,customers
variant,,,
control,168.145748,1271350.0,7561
treatment,169.223014,1258850.0,7439


### Descriptive guardrail rates by group

In [ ]:
guardrails = (
    df.groupby('variant')
      .agg(
          customers=('customer_id', 'count'),
          refunds=('refund', 'sum'),
          payment_failures=('payment_failed', 'sum'),
          support_contacts=('support_contact', 'sum')
      )
)

for measure in ['refunds', 'payment_failures', 'support_contacts']:
    guardrails[f'{measure}_rate'] = guardrails[measure] / guardrails['customers']

guardrails

,customers,refunds,payment_failures,support_contacts,refunds_rate,payment_failures_rate,support_contacts_rate
variant,,,,,,,
control,7561,36,146,316,0.004761,0.019310,0.041793
treatment,7439,41,152,308,0.005511,0.020433,0.041403
